## 5.1.1 Frame Transformations and Rigid Registration Tests

In [10]:
import sys
from pathlib import Path

import numpy as np
from IPython.display import display, Markdown

# Make sure the assignment root is importable
root = Path.cwd().resolve()
if not (root / "src").is_dir():
    root = root.parent
sys.path.insert(0, str(root))

from src import vct3, Rot, Frame, uvct3, simulation


# ============================================================
# Helper functions
# ============================================================

test_results_511 = []

def add_result(test_id, description, observed, criterion, passed):
    test_results_511.append({
        "ID": test_id,
        "Test": description,
        "Observed": observed,
        "Criterion": criterion,
        "Result": "PASS" if passed else "FAIL"
    })


def registration_errors(F_est, F_true):
    """Translation error (mm) and rotation error (degrees)."""

    t_error = (F_est.p - F_true.p).norm()

    R_delta = F_est.R.matrix @ F_true.R.matrix.T
    c = (np.trace(R_delta) - 1.0) / 2.0
    c = np.clip(c, -1.0, 1.0)

    r_error = np.degrees(np.arccos(c))
    return t_error, r_error


def make_observations(points, F, noise_std=0.0, seed=123):
    """Generate synthetic marker observations from known geometry."""

    rng = np.random.default_rng(seed)
    observations = []

    for a in points:
        b = F * a
        xyz = np.asarray(b.vec).ravel()

        if noise_std > 0:
            xyz = xyz + rng.normal(0, noise_std, size=3)

        observations.append(uvct3(vct3(*xyz)))

    return observations


# ============================================================
# UT-01: Frame inverse
# ============================================================

F = Frame(
    Rot.xyz_deg(20, -15, 35),
    vct3(100, 200, 300)
)

p = vct3(25, -40, 70)

p_recovered = F.inv() * (F * p)
error = (p_recovered - p).norm()

add_result(
    "UT-01", "Frame inverse",
    f"{error:.3e} mm",
    "< 1e-8 mm",
    error < 1e-8
)


# ============================================================
# UT-02: Frame composition with known expected coordinates
# ============================================================

F_AB = Frame(
    Rot.xyz_deg(0, 0, 90),
    vct3(100, 0, 0)
)

F_BC = Frame(
    Rot.eye(),
    vct3(0, 10, 0)
)

p_C = vct3(2, 0, 0)

# Expected:
# p_B = (2, 10, 0)
# Rotate +90 deg around Z and translate (100, 0, 0)
# p_A = (90, 2, 0)

p_expected = vct3(90, 2, 0)

p_composed = (F_AB * F_BC) * p_C
p_sequential = F_AB * (F_BC * p_C)

error = max(
    (p_composed - p_expected).norm(),
    (p_sequential - p_expected).norm()
)

add_result(
    "UT-02", "Frame composition",
    f"{error:.3e} mm",
    "< 1e-8 mm",
    error < 1e-8
)


# ============================================================
# UT-03: Tracker-to-Calibrator frame transformation
# ============================================================

# Known workspace placements
F_WT = Frame(
    Rot.xyz_deg(10, 20, -15),
    vct3(50, 20, 30)
)

F_WP = Frame(
    Rot.eye(),
    vct3(100, 200, 300)
)

F_WC = Frame(
    Rot.eye(),
    vct3(80, 150, 200)
)

p_tip_P = vct3(0, 0, -150)

# Convert pointer/calibrator frames into tracker frame
F_TP = F_WT.inv() * F_WP
F_TC = F_WT.inv() * F_WC

# Predict tip position in calibrator frame
p_tip_C = F_TC.inv() * F_TP * p_tip_P

# Independently known expected coordinates:
# Workspace tip = (100, 200, 150)
# Calibrator origin = (80, 150, 200)
# Relative tip = (20, 50, -50)
p_expected = vct3(20, 50, -50)

error = (p_tip_C - p_expected).norm()

add_result(
    "UT-03", "Tracker-to-calibrator",
    f"{error:.3e} mm",
    "< 1e-8 mm",
    error < 1e-8
)


# ============================================================
# Rigid registration setup
# ============================================================

tracker = simulation.create_tracker(
    "unit_test_registration",
    Frame.eye()
)

marker_points = [
    vct3(0, 0, 0),
    vct3(120, 0, 0),
    vct3(0, 90, 0),
    vct3(0, 0, 70),
    vct3(80, 60, 50)
]

F_true = Frame(
    Rot.xyz_deg(20, -15, 30),
    vct3(80, -45, 120)
)


def test_registration(test_id, points, noise_std,
                      t_tol, r_tol, description):

    observations = make_observations(
        points, F_true, noise_std=noise_std
    )

    F_est = tracker.procrustes_solver(points, observations)

    t_err, r_err = registration_errors(F_est, F_true)

    passed = (t_err < t_tol) and (r_err < r_tol)

    add_result(
        test_id,
        description,
        f"t={t_err:.4f} mm, R={r_err:.4f} deg",
        f"t<{t_tol:g}, R<{r_tol:g}",
        passed
    )


# UT-04: Minimum noncollinear correspondence set
test_registration(
    "UT-04", marker_points[:3], 0.0,
    1e-6, 1e-4,
    "Exact registration, 3 markers"
)

# UT-05: Larger 3D marker constellation
test_registration(
    "UT-05", marker_points, 0.0,
    1e-6, 1e-4,
    "Exact registration, 5 markers"
)

# UT-06: Small synthetic measurement noise
test_registration(
    "UT-06", marker_points, 0.05,
    0.5, 0.5,
    "Noisy registration, sigma=0.05 mm"
)

# UT-07: Larger synthetic measurement noise
test_registration(
    "UT-07", marker_points, 0.20,
    1.0, 1.0,
    "Noisy registration, sigma=0.20 mm"
)


# ============================================================
# UT-08: Insufficient marker correspondences
# ============================================================

try:
    tracker.procrustes_solver(
        marker_points[:2],
        make_observations(marker_points[:2], F_true)
    )
    rejected = False
except ValueError:
    rejected = True

add_result(
    "UT-08",
    "Only 2 markers",
    "ValueError" if rejected else "No exception",
    "ValueError",
    rejected
)


# ============================================================
# UT-09: Mismatched correspondence counts
# ============================================================

try:
    tracker.procrustes_solver(
        marker_points[:4],
        make_observations(marker_points[:3], F_true)
    )
    rejected = False
except ValueError:
    rejected = True

add_result(
    "UT-09",
    "Mismatched marker counts",
    "ValueError" if rejected else "No exception",
    "ValueError",
    rejected
)


# ============================================================
# Display a compact unit-test result table
# ============================================================

lines = [
    "| ID | Test | Observed result | Pass criterion | Status |",
    "|---|---|---|---|---|"
]

for result in test_results_511:
    lines.append(
        f"| {result['ID']} "
        f"| {result['Test']} "
        f"| {result['Observed']} "
        f"| {result['Criterion']} "
        f"| {result['Result']} |"
    )

display(Markdown("### 5.1.1 Unit Test Results\n\n" + "\n".join(lines)))

n_pass = sum(r["Result"] == "PASS" for r in test_results_511)
n_total = len(test_results_511)

print(f"\nSummary: {n_pass}/{n_total} tests passed")

assert n_pass == n_total, "Some unit tests failed"


### 5.1.1 Unit Test Results

| ID | Test | Observed result | Pass criterion | Status |
|---|---|---|---|---|
| UT-01 | Frame inverse | 0.000e+00 mm | < 1e-8 mm | PASS |
| UT-02 | Frame composition | 4.441e-16 mm | < 1e-8 mm | PASS |
| UT-03 | Tracker-to-calibrator | 4.278e-14 mm | < 1e-8 mm | PASS |
| UT-04 | Exact registration, 3 markers | t=0.0000 mm, R=0.0000 deg | t<1e-06, R<0.0001 | PASS |
| UT-05 | Exact registration, 5 markers | t=0.0000 mm, R=0.0000 deg | t<1e-06, R<0.0001 | PASS |
| UT-06 | Noisy registration, sigma=0.05 mm | t=0.0223 mm, R=0.0215 deg | t<0.5, R<0.5 | PASS |
| UT-07 | Noisy registration, sigma=0.20 mm | t=0.0892 mm, R=0.0860 deg | t<1, R<1 | PASS |
| UT-08 | Only 2 markers | ValueError | ValueError | PASS |
| UT-09 | Mismatched marker counts | ValueError | ValueError | PASS |


Summary: 9/9 tests passed


## 5.1.2 Marker Geometry Refinement Tests

In [11]:
import numpy as np
from types import SimpleNamespace
from unittest.mock import patch
from IPython.display import display, Markdown

from src import vct3, Rot, Frame, uvct3, simulation
from functions.marker_refinement import refine_marker_body


# ------------------------------------------------------
# Synthetic setup
# ------------------------------------------------------

base_points = np.array([
    [0.0, 0.0, 0.0],
    [120.0, 0.0, 0.0],
    [0.0, 90.0, 0.0],
    [0.0, 0.0, 70.0]
])

F1 = Frame(Rot.xyz_deg(20, -10, 30), vct3(100, 40, 80))
F2 = Frame(Rot.xyz_deg(-15, 25, 5), vct3(80, -30, 110))
F3 = Frame(Rot.xyz_deg(30, 15, -20), vct3(50, 60, 90))


class TestMarkerBody:
    def __init__(self):
        self.name = "test_mb"
        self.nominal_marker_positions = [
            vct3(*row) for row in base_points
        ]
        self.update_history = []

    def set_marker_positions(self, positions):
        self.nominal_marker_positions = positions
        self.update_history.append(
            np.array([p.vec.ravel() for p in positions])
        )
        return self


def run_refinement_test(snapshots, frames, n_readings, passes):
    """
    Run the real refinement function using controlled observations.
    snapshots: local marker positions for each simulated reading.
    frames: known marker-body frame for each reading.
    """
    body = TestMarkerBody()
    placed = SimpleNamespace(body=body)

    assert len(snapshots) == len(frames) == n_readings * passes

    observations = []

    for local_positions, F in zip(snapshots, frames):
        markers = [
            uvct3(F * vct3(*p))
            for p in local_positions
        ]

        observations.append({
            body.name: SimpleNamespace(frame=F, markers=markers)
        })

    with patch.object(
        simulation,
        "sample_marker_bodies",
        side_effect=observations
    ) as mock_sample:

        refine_marker_body(
            trk=None,
            placed=placed,
            n_readings=n_readings,
            passes=passes
        )

        calls = mock_sample.call_count

    final_points = np.array([
        p.vec.ravel()
        for p in body.nominal_marker_positions
    ])

    return body, final_points, calls


results_512 = []

def record_test(test_id, name, observed, criterion, passed):
    results_512.append({
        "ID": test_id,
        "Test": name,
        "Observed": observed,
        "Criterion": criterion,
        "Status": "PASS" if passed else "FAIL"
    })


# ------------------------------------------------------
# UT-10: Exact geometry recovery
# ------------------------------------------------------

body, estimated, calls = run_refinement_test(
    snapshots=[base_points.copy() for _ in range(3)],
    frames=[F1, F2, F3],
    n_readings=3,
    passes=1
)

error = np.max(np.linalg.norm(estimated - base_points, axis=1))

record_test(
    "UT-10", "Exact geometry recovery",
    f"{error:.3e} mm",
    "Error < 1e-8 mm",
    error < 1e-8
)


# ------------------------------------------------------
# UT-11: Averaging symmetric observation noise
# ------------------------------------------------------

delta = np.array([0.6, -0.4, 0.2])

body, estimated, calls = run_refinement_test(
    snapshots=[
        base_points + delta,
        base_points - delta,
        base_points.copy()
    ],
    frames=[F1, F2, F3],
    n_readings=3,
    passes=1
)

error = np.max(np.linalg.norm(estimated - base_points, axis=1))

record_test(
    "UT-11", "Symmetric noise averaging",
    f"{error:.3e} mm",
    "Error < 1e-8 mm",
    error < 1e-8
)


# ------------------------------------------------------
# UT-12: Multi-pass updates
# ------------------------------------------------------

offsets = [
    np.array([0.2, 0.0, 0.0]),
    np.array([0.4, 0.0, 0.0]),
    np.array([-0.3, 0.0, 0.0]),
    np.array([-0.5, 0.0, 0.0])
]

body, estimated, calls = run_refinement_test(
    snapshots=[base_points + d for d in offsets],
    frames=[F1, F2, F1, F2],
    n_readings=2,
    passes=2
)

# First pass averages to +0.3 mm.
expected_first = base_points + np.array([0.3, 0, 0])

# Second pass averages to -0.4 mm.
expected_second = base_points + np.array([-0.4, 0, 0])

first_error = np.max(
    np.linalg.norm(body.update_history[0] - expected_first, axis=1)
)

final_error = np.max(
    np.linalg.norm(estimated - expected_second, axis=1)
)

passed = (
    len(body.update_history) == 2
    and calls == 4
    and first_error < 1e-8
    and final_error < 1e-8
)

record_test(
    "UT-12", "Multi-pass update",
    f"Updates={len(body.update_history)}, reads={calls}",
    "2 updates, 4 reads; exact means",
    passed
)


# ------------------------------------------------------
# UT-13: Marker indexing and correspondence
# ------------------------------------------------------

marker_offsets = np.array([
    [0.1, 0.2, 0.3],
    [-0.2, 0.1, -0.1],
    [0.0, -0.4, 0.2],
    [0.3, 0.1, -0.2]
])

expected = base_points + marker_offsets

body, estimated, calls = run_refinement_test(
    snapshots=[expected],
    frames=[F1],
    n_readings=1,
    passes=1
)

error = np.max(np.linalg.norm(estimated - expected, axis=1))

passed = (
    len(body.nominal_marker_positions) == len(base_points)
    and error < 1e-8
)

record_test(
    "UT-13", "Marker correspondence",
    f"{len(estimated)} markers, error={error:.3e} mm",
    "4 markers, correct order",
    passed
)


# ------------------------------------------------------
# UT-14: Zero-pass behavior
# ------------------------------------------------------

body, estimated, calls = run_refinement_test(
    snapshots=[],
    frames=[],
    n_readings=3,
    passes=0
)

unchanged = np.allclose(estimated, base_points)

passed = (
    unchanged
    and calls == 0
    and len(body.update_history) == 0
)

record_test(
    "UT-14", "Zero-pass behavior",
    f"Updates={len(body.update_history)}, reads={calls}",
    "0 updates, 0 reads",
    passed
)


# ------------------------------------------------------
# Summary table
# ------------------------------------------------------

lines = [
    "| ID | Test | Observed Result | Criterion | Status |",
    "|---|---|---|---|---|"
]

for r in results_512:
    lines.append(
        f"| {r['ID']} | {r['Test']} | {r['Observed']} "
        f"| {r['Criterion']} | {r['Status']} |"
    )

display(Markdown(
    "### 5.1.2 Unit Test Results\n\n" + "\n".join(lines)
))

n_pass = sum(r["Status"] == "PASS" for r in results_512)
n_total = len(results_512)

print(f"\nSummary: {n_pass}/{n_total} tests passed")

assert n_pass == n_total, "Some refinement unit tests failed"


### 5.1.2 Unit Test Results

| ID | Test | Observed Result | Criterion | Status |
|---|---|---|---|---|
| UT-10 | Exact geometry recovery | 1.460e-14 mm | Error < 1e-8 mm | PASS |
| UT-11 | Symmetric noise averaging | 1.446e-14 mm | Error < 1e-8 mm | PASS |
| UT-12 | Multi-pass update | Updates=2, reads=4 | 2 updates, 4 reads; exact means | PASS |
| UT-13 | Marker correspondence | 4 markers, error=3.362e-14 mm | 4 markers, correct order | PASS |
| UT-14 | Zero-pass behavior | Updates=0, reads=0 | 0 updates, 0 reads | PASS |


Summary: 5/5 tests passed


## 5.1.3 Calibration Pose Generation and Rotation Diversity Tests

In [12]:
import sys
from pathlib import Path

import numpy as np
from IPython.display import display, Markdown

# Make assignment root importable
root = Path.cwd().resolve()
if not (root / "functions").is_dir():
    root = root.parent

assert (root / "functions").is_dir(), "Assignment root not found"

if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from src import Rot

from functions.calibration_pose import (
    rotation_distance_deg,
    is_rotation_diverse,
    sample_diverse_rotation
)


# ============================================================
# Helper: record unit test results
# ============================================================

results_513 = []

def record_513(test_id, name, observed, criterion, passed):
    results_513.append({
        "ID": test_id,
        "Test": name,
        "Observed": observed,
        "Criterion": criterion,
        "Status": "PASS" if passed else "FAIL"
    })


# ============================================================
# UT-15: Identity rotation
# ============================================================

R = Rot.xyz_deg(20, -15, 30)

distance = rotation_distance_deg(R, R)

passed = abs(distance) < 1e-4

record_513(
    "UT-15",
    "Identity rotation distance",
    f"{distance:.8f} deg",
    "Distance < 1e-4 deg",
    passed
)


# ============================================================
# UT-16: Known rotation angle
# ============================================================

R1 = Rot.eye()
R2 = Rot.xyz_deg(90, 0, 0)

distance = rotation_distance_deg(R1, R2)

error = abs(distance - 90.0)

record_513(
    "UT-16",
    "Known 90-degree rotation",
    f"{distance:.6f} deg",
    "Error < 1e-4 deg",
    error < 1e-4
)


# ============================================================
# UT-17: Rotation distance symmetry
# ============================================================

R1 = Rot.xyz_deg(20, -10, 30)
R2 = Rot.xyz_deg(-40, 15, 25)

d12 = rotation_distance_deg(R1, R2)
d21 = rotation_distance_deg(R2, R1)

error = abs(d12 - d21)

record_513(
    "UT-17",
    "Rotation distance symmetry",
    f"Difference={error:.3e} deg",
    "Difference < 1e-6 deg",
    error < 1e-6
)


# ============================================================
# UT-18: Random Euler angle range
# ============================================================

rng = np.random.default_rng(123)

angles_list = []

for _ in range(30):
    R, angles = sample_diverse_rotation(
        rng,
        accepted_rotations=[],
        min_angle_diff=25.0
    )
    angles_list.append(angles)

angles_array = np.array(angles_list)

min_angle = np.min(angles_array)
max_angle = np.max(angles_array)

passed = np.all(
    (angles_array >= -70) & (angles_array <= 70)
)

record_513(
    "UT-18",
    "Euler angle sampling range",
    f"Observed range=[{min_angle}, {max_angle}] deg",
    "All angles in [-70, 70]",
    passed
)


# ============================================================
# UT-19: Rotation diversity enforcement
# ============================================================

rng = np.random.default_rng(123)

accepted = []

for _ in range(30):

    R, angles = sample_diverse_rotation(
        rng,
        accepted,
        min_angle_diff=25.0
    )

    accepted.append(R)

# Compute the minimum distance among all distinct pairs
pairwise_distances = []

for i in range(len(accepted)):
    for j in range(i + 1, len(accepted)):

        distance = rotation_distance_deg(
            accepted[i],
            accepted[j]
        )

        pairwise_distances.append(distance)

minimum_distance = min(pairwise_distances)

# Explicit rejection / acceptance checks
R_identity = Rot.eye()

R_too_close = Rot.xyz_deg(10, 0, 0)
R_far_enough = Rot.xyz_deg(40, 0, 0)

rejects_close = not is_rotation_diverse(
    R_too_close, [R_identity], 25.0
)

accepts_far = is_rotation_diverse(
    R_far_enough, [R_identity], 25.0
)

passed = (
    minimum_distance >= 25.0 - 1e-6
    and rejects_close
    and accepts_far
)

record_513(
    "UT-19",
    "Minimum rotation diversity",
    f"Min pairwise={minimum_distance:.4f} deg",
    "All pairs >=25 deg; close rejected",
    passed
)


# ============================================================
# UT-20: Reproducible pose generation
# ============================================================

def generate_pose_sequence(seed, n=20):

    rng = np.random.default_rng(seed)

    accepted = []
    angles_sequence = []

    for _ in range(n):

        R, angles = sample_diverse_rotation(
            rng,
            accepted,
            min_angle_diff=25.0
        )

        accepted.append(R)
        angles_sequence.append(angles)

    return angles_sequence


sequence_A = generate_pose_sequence(42)
sequence_B = generate_pose_sequence(42)
sequence_C = generate_pose_sequence(43)

same_seed_equal = (sequence_A == sequence_B)
different_seed_different = (sequence_A != sequence_C)

passed = same_seed_equal and different_seed_different

record_513(
    "UT-20",
    "Random seed reproducibility",
    f"Same={same_seed_equal}, Different={different_seed_different}",
    "Same seed identical; other seed differs",
    passed
)


# ============================================================
# Display results
# ============================================================

lines = [
    "| ID | Test | Observed Result | Criterion | Status |",
    "|---|---|---|---|---|"
]

for r in results_513:
    lines.append(
        f"| {r['ID']} "
        f"| {r['Test']} "
        f"| {r['Observed']} "
        f"| {r['Criterion']} "
        f"| {r['Status']} |"
    )

display(Markdown(
    "### 5.1.3 Unit Test Results\n\n"
    + "\n".join(lines)
))

n_pass = sum(
    r["Status"] == "PASS"
    for r in results_513
)

n_total = len(results_513)

print(f"\nSummary: {n_pass}/{n_total} tests passed")

assert n_pass == n_total, "Some pose-generation tests failed"


### 5.1.3 Unit Test Results

| ID | Test | Observed Result | Criterion | Status |
|---|---|---|---|---|
| UT-15 | Identity rotation distance | 0.00000000 deg | Distance < 1e-4 deg | PASS |
| UT-16 | Known 90-degree rotation | 90.000000 deg | Error < 1e-4 deg | PASS |
| UT-17 | Rotation distance symmetry | Difference=0.000e+00 deg | Difference < 1e-6 deg | PASS |
| UT-18 | Euler angle sampling range | Observed range=[-69, 63] deg | All angles in [-70, 70] | PASS |
| UT-19 | Minimum rotation diversity | Min pairwise=26.7912 deg | All pairs >=25 deg; close rejected | PASS |
| UT-20 | Random seed reproducibility | Same=True, Different=True | Same seed identical; other seed differs | PASS |


Summary: 6/6 tests passed


## 5.1.4 Tip Calibration Solver Tests

In [13]:

import sys
from pathlib import Path
import numpy as np
from IPython.display import display, Markdown

root = Path.cwd().resolve()
if not (root / "functions").is_dir():
    root = root.parent

assert (root / "functions").is_dir(), "Assignment root not found"
sys.path.insert(0, str(root))

from src import vct3, Rot, Frame
from functions.tip_calibration import compute_pivot_calibration


# =====================================================
# Helpers
# =====================================================

results_514 = []

def record(test_id, name, observed, criterion, passed):
    results_514.append({
        "ID": test_id,
        "Test": name,
        "Observed": observed,
        "Criterion": criterion,
        "Status": "PASS" if passed else "FAIL"
    })


def arr(v):
    return np.asarray(v.vec, dtype=float).ravel()


def position_error(p1, p2):
    return np.linalg.norm(arr(p1) - arr(p2))


def make_frames(n):
    """Generate known pointer/calibrator poses."""
    F_ptr = []
    F_cal = []

    for j in range(n):
        F_ptr.append(Frame(
            Rot.xyz_deg(
                10 + 7*j, -20 + 3*j, 5 + 9*j
            ),
            vct3(80 + 2*j, -20 + j, 100 + 3*j)
        ))

        F_cal.append(Frame(
            Rot.xyz_deg(
                -15 + 2*j, 10 - j, 20 + 4*j
            ),
            vct3(-30 + 3*j, 40 - j, 20 + 2*j)
        ))

    return F_ptr, F_cal


def make_exact_sensor_readings(F_ptr, F_cal, p_true):
    """Generate exact sensor readings from a known tip."""
    return [
        Fc.inv() * Fp * p_true
        for Fp, Fc in zip(F_ptr, F_cal)
    ]


# =====================================================
# UT-21: Exact synthetic solutions
# =====================================================

cases = [
    (3,  vct3(0, 0, -150)),
    (10, vct3(5, -8, -120)),
    (30, vct3(100, 100, 250))
]

exact_errors = []

for n, p_true in cases:
    F_ptr, F_cal = make_frames(n)

    sensed = make_exact_sensor_readings(
        F_ptr, F_cal, p_true
    )

    p_est, cov = compute_pivot_calibration(
        F_ptr, F_cal, sensed
    )

    exact_errors.append(position_error(p_est, p_true))

max_exact_error = max(exact_errors)

record(
    "UT-21",
    "Exact synthetic solutions",
    f"N=3,10,30; max={max_exact_error:.3e} mm",
    "Max error < 1e-8 mm",
    max_exact_error < 1e-8
)


# =====================================================
# UT-22: Single valid observation
# =====================================================

p_true = vct3(10, -5, -150)
F_ptr, F_cal = make_frames(1)
sensed = make_exact_sensor_readings(F_ptr, F_cal, p_true)

p_est, cov = compute_pivot_calibration(
    F_ptr, F_cal, sensed
)

error = position_error(p_est, p_true)
cov_unavailable = np.isnan(cov).all()

record(
    "UT-22",
    "Single observation",
    f"Error={error:.3e} mm; Cov NaN={cov_unavailable}",
    "Exact tip; covariance unavailable",
    error < 1e-8 and cov_unavailable
)


# =====================================================
# UT-23: Controlled measurement errors
# =====================================================

p_true = vct3(10, -5, -150)
N = 4

F_ptr = [Frame.eye() for _ in range(N)]
F_cal = [Frame.eye() for _ in range(N)]

noise = np.array([
    [0.4,  0.0, -0.2],
    [0.2, -0.4,  0.0],
    [0.0,  0.2,  0.2],
    [0.2,  0.2,  0.0]
])

sensed = [
    vct3(*(arr(p_true) + noise[j]))
    for j in range(N)
]

p_est_noisy, cov_noisy = compute_pivot_calibration(
    F_ptr, F_cal, sensed
)

# For identity frames, least squares returns
# p_true + the mean measurement error
p_expected = vct3(*(arr(p_true) + np.mean(noise, axis=0)))

error = position_error(p_est_noisy, p_expected)

record(
    "UT-23",
    "Controlled measurement errors",
    f"Solution error={error:.3e} mm",
    "Matches analytical LS mean",
    error < 1e-8
)


# =====================================================
# UT-24: Covariance estimation
# =====================================================

# Independent analytical calculation for repeated identity A
mean_noise = np.mean(noise, axis=0)
rss = np.sum((noise - mean_noise) ** 2)

dof = 3*N - 3
sigma_squared = rss / dof

# A.T @ A = N * I
cov_expected = (sigma_squared / N) * np.eye(3)

cov_error = np.max(np.abs(cov_noisy - cov_expected))

passed = (
    np.all(np.isfinite(cov_noisy))
    and np.allclose(cov_noisy, cov_noisy.T)
    and cov_error < 1e-10
)

record(
    "UT-24",
    "Covariance estimation",
    f"Max difference={cov_error:.3e} mm^2",
    "Difference < 1e-10 mm^2",
    passed
)


# =====================================================
# UT-25: Invalid sensor readings are skipped
# =====================================================

p_true = vct3(10, -5, -150)
F_ptr, F_cal = make_frames(5)

sensed = make_exact_sensor_readings(
    F_ptr, F_cal, p_true
)

sensed[1] = None
sensed[3] = None

p_est, cov = compute_pivot_calibration(
    F_ptr, F_cal, sensed
)

error = position_error(p_est, p_true)

record(
    "UT-25",
    "Invalid sensor filtering",
    f"2 None values; error={error:.3e} mm",
    "Valid pairs retained; error < 1e-8",
    error < 1e-8
)


# =====================================================
# UT-26: No valid observations
# =====================================================

F_ptr, F_cal = make_frames(3)

try:
    compute_pivot_calibration(
        F_ptr, F_cal, [None, None, None]
    )
    rejected = False
except ValueError:
    rejected = True

record(
    "UT-26",
    "No valid observations",
    "ValueError" if rejected else "No exception",
    "ValueError",
    rejected
)


# =====================================================
# UT-27: Mismatched input lengths
# =====================================================

F_ptr, F_cal = make_frames(3)

try:
    compute_pivot_calibration(
        F_ptr,
        F_cal[:2],
        [vct3(0, 0, 0)] * 3
    )
    rejected = False
except ValueError:
    rejected = True

record(
    "UT-27",
    "Mismatched input lengths",
    "ValueError" if rejected else "No exception",
    "ValueError",
    rejected
)


# =====================================================
# UT-28: Noise scaling property
# =====================================================

# Double all measurement perturbations
sensed_double = [
    vct3(*(arr(p_true) + 2 * noise[j]))
    for j in range(N)
]

# Use the same identity frames from UT-23
F_identity = [Frame.eye() for _ in range(N)]

p_double, cov_double = compute_pivot_calibration(
    F_identity, F_identity, sensed_double
)

# Original noisy solution from UT-23
bias_original = arr(p_est_noisy) - arr(p_true)
bias_double = arr(p_double) - arr(p_true)

bias_difference = np.max(
    np.abs(bias_double - 2 * bias_original)
)

cov_difference = np.max(
    np.abs(cov_double - 4 * cov_noisy)
)

passed = (
    bias_difference < 1e-10
    and cov_difference < 1e-10
)

record(
    "UT-28",
    "Noise scaling",
    f"Bias diff={bias_difference:.3e}; "
    f"Cov diff={cov_difference:.3e}",
    "Bias x2; covariance x4",
    passed
)


# =====================================================
# Display summary
# =====================================================

lines = [
    "| ID | Test | Observed Result | Criterion | Status |",
    "|---|---|---|---|---|"
]

for r in results_514:
    lines.append(
        f"| {r['ID']} | {r['Test']} | {r['Observed']} "
        f"| {r['Criterion']} | {r['Status']} |"
    )

display(Markdown(
    "### 5.1.4 Tip Calibration Solver Results\n\n"
    + "\n".join(lines)
))

n_pass = sum(r["Status"] == "PASS" for r in results_514)

print(f"\nSummary: {n_pass}/{len(results_514)} tests passed")

assert n_pass == len(results_514), "Some solver tests failed"


### 5.1.4 Tip Calibration Solver Results

| ID | Test | Observed Result | Criterion | Status |
|---|---|---|---|---|
| UT-21 | Exact synthetic solutions | N=3,10,30; max=1.302e-13 mm | Max error < 1e-8 mm | PASS |
| UT-22 | Single observation | Error=1.697e-14 mm; Cov NaN=True | Exact tip; covariance unavailable | PASS |
| UT-23 | Controlled measurement errors | Solution error=0.000e+00 mm | Matches analytical LS mean | PASS |
| UT-24 | Covariance estimation | Max difference=2.394e-16 mm^2 | Difference < 1e-10 mm^2 | PASS |
| UT-25 | Invalid sensor filtering | 2 None values; error=4.469e-14 mm | Valid pairs retained; error < 1e-8 | PASS |
| UT-26 | No valid observations | ValueError | ValueError | PASS |
| UT-27 | Mismatched input lengths | ValueError | ValueError | PASS |
| UT-28 | Noise scaling | Bias diff=3.553e-15; Cov diff=1.221e-15 | Bias x2; covariance x4 | PASS |


Summary: 8/8 tests passed


## 5.1.5 Tip Position Prediction Tests

In [14]:

import sys
from pathlib import Path

import numpy as np
from IPython.display import display, Markdown

root = Path.cwd().resolve()
if not (root / "functions").is_dir():
    root = root.parent

assert (root / "functions").is_dir(), "Assignment root not found"

if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from src import vct3, Rot, Frame
from functions.tip_prediction import (
    predict_tip_position,
    position_error_mm,
    within_tolerance
)


# ============================================================
# Helpers
# ============================================================

results_515 = []

def record_515(test_id, test, observed, criterion, passed):
    results_515.append({
        "ID": test_id,
        "Test": test,
        "Observed": observed,
        "Criterion": criterion,
        "Status": "PASS" if passed else "FAIL"
    })


def xyz(v):
    return np.asarray(v.vec, dtype=float).ravel()


# ============================================================
# UT-29: Translation-only tip prediction
# ============================================================

F_ptr = Frame(
    Rot.eye(),
    vct3(100, 200, 300)
)

F_near = Frame(
    Rot.eye(),
    vct3(50, 100, 150)
)

p_tip = vct3(0, 0, -150)

# Analytical result:
# World tip = (100, 200, 150)
# Near origin = (50, 100, 150)
# Expected = (50, 100, 0)

p_expected = vct3(50, 100, 0)

p_est = predict_tip_position(
    F_ptr, F_near, p_tip
)

error = position_error_mm(p_est, p_expected)

record_515(
    "UT-29",
    "Translation-only prediction",
    f"Error={error:.3e} mm",
    "Error < 1e-8 mm",
    error < 1e-8
)


# ============================================================
# UT-30: Rotated reference frame
# ============================================================

F_ptr = Frame(
    Rot.eye(),
    vct3(20, 30, 50)
)

F_near = Frame(
    Rot.xyz_deg(0, 0, 90),
    vct3(10, 20, 30)
)

p_tip = vct3(10, 0, -10)

# Analytical calculation:
#
# Tracker-frame tip = (30, 30, 40)
# Subtract near-frame origin: (20, 10, 10)
# Rotate by -90 degrees about Z:
# Expected near-frame tip = (10, -20, 10)

p_expected = vct3(10, -20, 10)

p_est = predict_tip_position(
    F_ptr, F_near, p_tip
)

error = position_error_mm(p_est, p_expected)

record_515(
    "UT-30",
    "Rotated reference frame",
    f"Error={error:.3e} mm",
    "Error < 1e-8 mm",
    error < 1e-8
)


# ============================================================
# UT-31: Identical pointer and reference frames
# ============================================================

F_same = Frame(
    Rot.xyz_deg(25, -35, 40),
    vct3(300, -100, 250)
)

p_tip = vct3(5, -8, -150)

p_est = predict_tip_position(
    F_same, F_same, p_tip
)

error = position_error_mm(p_est, p_tip)

record_515(
    "UT-31",
    "Identical pointer/reference frames",
    f"Error={error:.3e} mm",
    "Prediction equals local tip",
    error < 1e-8
)


# ============================================================
# UT-32: Invariance to common tracker-frame change
# ============================================================

F_ptr = Frame(
    Rot.xyz_deg(20, -30, 10),
    vct3(100, 200, 300)
)

F_near = Frame(
    Rot.xyz_deg(-25, 5, 40),
    vct3(50, 100, 150)
)

p_tip = vct3(10, -5, -150)

# Original prediction
p_original = predict_tip_position(
    F_ptr, F_near, p_tip
)

# Change the common tracker coordinate frame
G = Frame(
    Rot.xyz_deg(10, -25, 35),
    vct3(400, -80, 150)
)

F_ptr_new = G * F_ptr
F_near_new = G * F_near

p_transformed = predict_tip_position(
    F_ptr_new, F_near_new, p_tip
)

error = position_error_mm(
    p_original, p_transformed
)

record_515(
    "UT-32",
    "Tracker-frame invariance",
    f"Difference={error:.3e} mm",
    "Difference < 1e-8 mm",
    error < 1e-8
)


# ============================================================
# UT-33: Known tip calibration error propagation
# ============================================================

F_ptr = Frame(
    Rot.xyz_deg(15, -30, 25),
    vct3(100, 200, 300)
)

F_near = Frame(
    Rot.xyz_deg(-10, 20, 35),
    vct3(50, 80, 120)
)

p_true = vct3(10, -5, -150)

# Inject known 1 mm calibration error:
# sqrt(0.6^2 + (-0.8)^2) = 1 mm

p_estimated_tip = vct3(10.6, -5.8, -150)

p_actual = predict_tip_position(
    F_ptr, F_near, p_true
)

p_predicted = predict_tip_position(
    F_ptr, F_near, p_estimated_tip
)

error = position_error_mm(p_predicted, p_actual)

difference = abs(error - 1.0)

record_515(
    "UT-33",
    "Known calibration error",
    f"Position error={error:.6f} mm",
    "Expected error = 1 mm",
    difference < 1e-8
)


# ============================================================
# UT-34: Euclidean position-error calculation
# ============================================================

p1 = vct3(3, -4, 12)
p2 = vct3(0, 0, 0)

# sqrt(3^2 + (-4)^2 + 12^2) = 13 mm
error_known = position_error_mm(p1, p2)

# Identical points must have zero error
error_zero = position_error_mm(p1, p1)

passed = (
    abs(error_known - 13.0) < 1e-8
    and abs(error_zero) < 1e-8
)

record_515(
    "UT-34",
    "Euclidean error metric",
    f"Known={error_known:.4f} mm; "
    f"Zero={error_zero:.4f} mm",
    "13 mm and 0 mm",
    passed
)


# ============================================================
# UT-35: 2 mm accuracy threshold boundary
# ============================================================

below = within_tolerance(1.9999)
at_boundary = within_tolerance(2.0)
above = within_tolerance(2.0001)

passed = (
    below is True
    and at_boundary is True
    and above is False
)

record_515(
    "UT-35",
    "2 mm threshold boundary",
    f"Below={below}, At={at_boundary}, Above={above}",
    "True, True, False",
    passed
)


# ============================================================
# Results table
# ============================================================

lines = [
    "| ID | Test | Observed Result | Criterion | Status |",
    "|---|---|---|---|---|"
]

for r in results_515:
    lines.append(
        f"| {r['ID']} | {r['Test']} "
        f"| {r['Observed']} "
        f"| {r['Criterion']} "
        f"| {r['Status']} |"
    )

display(Markdown(
    "### 5.1.5 Tip Position Prediction Results\n\n"
    + "\n".join(lines)
))

n_pass = sum(
    r["Status"] == "PASS"
    for r in results_515
)

n_total = len(results_515)

print(f"\nSummary: {n_pass}/{n_total} tests passed")

assert n_pass == n_total, "Some tip prediction tests failed"


### 5.1.5 Tip Position Prediction Results

| ID | Test | Observed Result | Criterion | Status |
|---|---|---|---|---|
| UT-29 | Translation-only prediction | Error=0.000e+00 mm | Error < 1e-8 mm | PASS |
| UT-30 | Rotated reference frame | Error=0.000e+00 mm | Error < 1e-8 mm | PASS |
| UT-31 | Identical pointer/reference frames | Error=9.441e-15 mm | Prediction equals local tip | PASS |
| UT-32 | Tracker-frame invariance | Difference=5.186e-14 mm | Difference < 1e-8 mm | PASS |
| UT-33 | Known calibration error | Position error=1.000000 mm | Expected error = 1 mm | PASS |
| UT-34 | Euclidean error metric | Known=13.0000 mm; Zero=0.0000 mm | 13 mm and 0 mm | PASS |
| UT-35 | 2 mm threshold boundary | Below=True, At=True, Above=False | True, True, False | PASS |


Summary: 7/7 tests passed
